In [10]:
from wds_astropy_table import parse_wdsweb
from wds_astropy_table import parse_sixth_orbital

from astropy.table import Table
from astropy.io import ascii

import numpy as np
import sys

table_wds = parse_wdsweb('wdsweb_summ2.txt')
#table_orbital = parse_sixth_orbital('orb6orbits.txt')



In [11]:
discoverer="MLB 115"
#wds_id="04153-0739"
components="AB"

normalized=discoverer.replace(" ", "")+components
wds = table_wds[(table_wds['discoverer_normalized']==normalized) & (table_wds['components']==components)][0]
#wds = table_wds[(table_wds['id']==wds_id) & (table_wds['components']==components)][0]
wds

id,discoverer,components,first_date,last_date,num_obs,first_pa,last_pa,first_sep,last_sep,pri_mag,sec_mag,spectral,pri_pm_ra,pri_pm_dec,sec_pm_ra,sec_pm_dec,dm,notes,j2000,discoverer_normalized
str10,str7,str5,int64,int64,int64,int64,int64,float64,float64,float64,float64,str9,float64,float64,float64,float64,str8,str4,str18,str12
03162+5810,MLB 115,AB,1914,2020,107,39,0,2.8,5.1,10.3,11.38,M2V~:,431.0,-325.0,436.0,-305.0,,O,031614.02+581001.4,MLB115AB


In [12]:
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

# "wds coordinate string"
wcs = wds['j2000']

# parse into format recognized by skycoord
formatted_coord_string = wcs[0:2]+'h'+wcs[2:4]+'m'+wcs[4:9]+'s' + ' ' + wcs[9:12]+'d'+wcs[12:14]+'m'+wcs[14:18]+'s'
print(wcs,"=>", formatted_coord_string)

# get the position angle and offset of secondary
pa = wds['last_pa']*u.deg
sep = wds['last_sep']*u.arcsec

# create skycoord objects
c_pri_j2000 = SkyCoord(formatted_coord_string, unit=(u.hourangle, u.deg), obstime=Time('J2000.0'))
c_sec_j2000 = c_pri_j2000.directional_offset_by(pa,sep)
print(c_pri_j2000.separation(c_sec_j2000))
print(c_pri_j2000.position_angle(c_sec_j2000).to(u.deg))

# adjust to gaia epoch
c_pri = c_pri_j2000.transform_to(SkyCoord(ra=0*u.deg, dec=0*u.deg, frame='icrs', equinox='J2016.0'))
c_sec = c_sec_j2000.transform_to(SkyCoord(ra=0*u.deg, dec=0*u.deg, frame='icrs', equinox='J2016.0'))

# Display the coordinates in ICRS (Gaia standard)
print(f"Primary j2000:   {c_pri.to_string('hmsdms')}")
print(f"Secondary j2000: {c_sec.to_string('hmsdms')}")
print(f"Primary j2016:   {c_pri.to_string('hmsdms')}")
print(f"Secondary j2016: {c_sec.to_string('hmsdms')}")


031614.02+581001.4 => 03h16m14.02s +58d10m01.4s
0d00m05.1s
0d00m00s
Primary j2000:   03h16m14.02s +58d10m01.4s
Secondary j2000: 03h16m14.02s +58d10m06.5s
Primary j2016:   03h16m14.02s +58d10m01.4s
Secondary j2016: 03h16m14.02s +58d10m06.5s


In [15]:
# do a Gaia cone search for the primary and secondary
from astroquery.gaia import Gaia

search_radius = 8*u.arcsec

job = Gaia.cone_search_async(c_pri, radius=search_radius)
results_pri = job.get_results()

job = Gaia.cone_search_async(c_sec, radius=search_radius)
results_sec = job.get_results()

display(results_pri)
display(results_sec)
#results_t = Table()
#results_t = vstack([results_t, results])

print(f"Primary source_id={results_pri['source_id'][0]}")
print(f"Secondary source_id={results_sec['source_id'][0]}")

INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]


solution_id,designation,source_id,random_index,ref_epoch,ra,ra_error,dec,dec_error,parallax,parallax_error,parallax_over_error,pm,pmra,pmra_error,pmdec,pmdec_error,ra_dec_corr,ra_parallax_corr,ra_pmra_corr,ra_pmdec_corr,dec_parallax_corr,dec_pmra_corr,dec_pmdec_corr,parallax_pmra_corr,parallax_pmdec_corr,pmra_pmdec_corr,astrometric_n_obs_al,astrometric_n_obs_ac,astrometric_n_good_obs_al,astrometric_n_bad_obs_al,astrometric_gof_al,astrometric_chi2_al,astrometric_excess_noise,astrometric_excess_noise_sig,astrometric_params_solved,astrometric_primary_flag,nu_eff_used_in_astrometry,pseudocolour,pseudocolour_error,ra_pseudocolour_corr,dec_pseudocolour_corr,parallax_pseudocolour_corr,pmra_pseudocolour_corr,pmdec_pseudocolour_corr,astrometric_matched_transits,visibility_periods_used,astrometric_sigma5d_max,matched_transits,new_matched_transits,matched_transits_removed,ipd_gof_harmonic_amplitude,ipd_gof_harmonic_phase,ipd_frac_multi_peak,ipd_frac_odd_win,ruwe,scan_direction_strength_k1,scan_direction_strength_k2,scan_direction_strength_k3,scan_direction_strength_k4,scan_direction_mean_k1,scan_direction_mean_k2,scan_direction_mean_k3,scan_direction_mean_k4,duplicated_source,phot_g_n_obs,phot_g_mean_flux,phot_g_mean_flux_error,phot_g_mean_flux_over_error,phot_g_mean_mag,phot_bp_n_obs,phot_bp_mean_flux,phot_bp_mean_flux_error,phot_bp_mean_flux_over_error,phot_bp_mean_mag,phot_rp_n_obs,phot_rp_mean_flux,phot_rp_mean_flux_error,phot_rp_mean_flux_over_error,phot_rp_mean_mag,phot_bp_rp_excess_factor,phot_bp_n_contaminated_transits,phot_bp_n_blended_transits,phot_rp_n_contaminated_transits,phot_rp_n_blended_transits,phot_proc_mode,bp_rp,bp_g,g_rp,radial_velocity,radial_velocity_error,rv_method_used,rv_nb_transits,rv_nb_deblended_transits,rv_visibility_periods_used,rv_expected_sig_to_noise,rv_renormalised_gof,rv_chisq_pvalue,rv_time_duration,rv_amplitude_robust,rv_template_teff,rv_template_logg,rv_template_fe_h,rv_atm_param_origin,vbroad,vbroad_error,vbroad_nb_transits,grvs_mag,grvs_mag_error,grvs_mag_nb_transits,rvs_spec_sig_to_noise,phot_variable_flag,l,b,ecl_lon,ecl_lat,in_qso_candidates,in_galaxy_candidates,non_single_star,has_xp_continuous,has_xp_sampled,has_rvs,has_epoch_photometry,has_epoch_rv,has_mcmc_gspphot,has_mcmc_msc,in_andromeda_survey,classprob_dsc_combmod_quasar,classprob_dsc_combmod_galaxy,classprob_dsc_combmod_star,teff_gspphot,teff_gspphot_lower,teff_gspphot_upper,logg_gspphot,logg_gspphot_lower,logg_gspphot_upper,mh_gspphot,mh_gspphot_lower,mh_gspphot_upper,distance_gspphot,distance_gspphot_lower,distance_gspphot_upper,azero_gspphot,azero_gspphot_lower,azero_gspphot_upper,ag_gspphot,ag_gspphot_lower,ag_gspphot_upper,ebpminrp_gspphot,ebpminrp_gspphot_lower,ebpminrp_gspphot_upper,libname_gspphot,dist
,,,,yr,deg,mas,deg,mas,mas,mas,,mas / yr,mas / yr,mas / yr,mas / yr,mas / yr,,,,,,,,,,,,,,,,,mas,,,,1 / um,1 / um,1 / um,,,,,,,,mas,,,,,deg,,,,,,,,deg,deg,deg,deg,,,electron / s,electron / s,,mag,,electron / s,electron / s,,mag,,electron / s,electron / s,,mag,,,,,,,mag,mag,mag,km / s,km / s,,,,,,,,d,km / s,K,log(cm.s**-2),dex,,km / s,km / s,,mag,mag,,,,deg,deg,deg,deg,,,,,,,,,,,,,,,K,K,K,log(cm.s**-2),log(cm.s**-2),log(cm.s**-2),dex,dex,dex,pc,pc,pc,mag,mag,mag,mag,mag,mag,mag,mag,mag,,
int64,object,int64,int64,float64,float64,float32,float64,float32,float64,float32,float32,float32,float64,float32,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int16,int16,int16,int16,float32,float32,float32,float32,int16,bool,float32,float32,float32,float32,float32,float32,float32,float32,int16,int16,float32,int16,int16,int16,float32,float32,int16,int16,float32,float32,float32,float32,float32,float32,float32,float32,float32,bool,int16,float64,float32,float32,float32,int16,float64,float32,float32,float32,int16,float64,float32,float32,float32,float32,int16,int16,int16,int16,int16,float32,float32,float32,float32,float32,int16,int16,int16,int16,float32,float32,float32,float32,float32,float32,float32,fl

solution_id,designation,source_id,random_index,ref_epoch,ra,ra_error,dec,dec_error,parallax,parallax_error,parallax_over_error,pm,pmra,pmra_error,pmdec,pmdec_error,ra_dec_corr,ra_parallax_corr,ra_pmra_corr,ra_pmdec_corr,dec_parallax_corr,dec_pmra_corr,dec_pmdec_corr,parallax_pmra_corr,parallax_pmdec_corr,pmra_pmdec_corr,astrometric_n_obs_al,astrometric_n_obs_ac,astrometric_n_good_obs_al,astrometric_n_bad_obs_al,astrometric_gof_al,astrometric_chi2_al,astrometric_excess_noise,astrometric_excess_noise_sig,astrometric_params_solved,astrometric_primary_flag,nu_eff_used_in_astrometry,pseudocolour,pseudocolour_error,ra_pseudocolour_corr,dec_pseudocolour_corr,parallax_pseudocolour_corr,pmra_pseudocolour_corr,pmdec_pseudocolour_corr,astrometric_matched_transits,visibility_periods_used,astrometric_sigma5d_max,matched_transits,new_matched_transits,matched_transits_removed,ipd_gof_harmonic_amplitude,ipd_gof_harmonic_phase,ipd_frac_multi_peak,ipd_frac_odd_win,ruwe,scan_direction_strength_k1,scan_direction_strength_k2,scan_direction_strength_k3,scan_direction_strength_k4,scan_direction_mean_k1,scan_direction_mean_k2,scan_direction_mean_k3,scan_direction_mean_k4,duplicated_source,phot_g_n_obs,phot_g_mean_flux,phot_g_mean_flux_error,phot_g_mean_flux_over_error,phot_g_mean_mag,phot_bp_n_obs,phot_bp_mean_flux,phot_bp_mean_flux_error,phot_bp_mean_flux_over_error,phot_bp_mean_mag,phot_rp_n_obs,phot_rp_mean_flux,phot_rp_mean_flux_error,phot_rp_mean_flux_over_error,phot_rp_mean_mag,phot_bp_rp_excess_factor,phot_bp_n_contaminated_transits,phot_bp_n_blended_transits,phot_rp_n_contaminated_transits,phot_rp_n_blended_transits,phot_proc_mode,bp_rp,bp_g,g_rp,radial_velocity,radial_velocity_error,rv_method_used,rv_nb_transits,rv_nb_deblended_transits,rv_visibility_periods_used,rv_expected_sig_to_noise,rv_renormalised_gof,rv_chisq_pvalue,rv_time_duration,rv_amplitude_robust,rv_template_teff,rv_template_logg,rv_template_fe_h,rv_atm_param_origin,vbroad,vbroad_error,vbroad_nb_transits,grvs_mag,grvs_mag_error,grvs_mag_nb_transits,rvs_spec_sig_to_noise,phot_variable_flag,l,b,ecl_lon,ecl_lat,in_qso_candidates,in_galaxy_candidates,non_single_star,has_xp_continuous,has_xp_sampled,has_rvs,has_epoch_photometry,has_epoch_rv,has_mcmc_gspphot,has_mcmc_msc,in_andromeda_survey,classprob_dsc_combmod_quasar,classprob_dsc_combmod_galaxy,classprob_dsc_combmod_star,teff_gspphot,teff_gspphot_lower,teff_gspphot_upper,logg_gspphot,logg_gspphot_lower,logg_gspphot_upper,mh_gspphot,mh_gspphot_lower,mh_gspphot_upper,distance_gspphot,distance_gspphot_lower,distance_gspphot_upper,azero_gspphot,azero_gspphot_lower,azero_gspphot_upper,ag_gspphot,ag_gspphot_lower,ag_gspphot_upper,ebpminrp_gspphot,ebpminrp_gspphot_lower,ebpminrp_gspphot_upper,libname_gspphot,dist
,,,,yr,deg,mas,deg,mas,mas,mas,,mas / yr,mas / yr,mas / yr,mas / yr,mas / yr,,,,,,,,,,,,,,,,,mas,,,,1 / um,1 / um,1 / um,,,,,,,,mas,,,,,deg,,,,,,,,deg,deg,deg,deg,,,electron / s,electron / s,,mag,,electron / s,electron / s,,mag,,electron / s,electron / s,,mag,,,,,,,mag,mag,mag,km / s,km / s,,,,,,,,d,km / s,K,log(cm.s**-2),dex,,km / s,km / s,,mag,mag,,,,deg,deg,deg,deg,,,,,,,,,,,,,,,K,K,K,log(cm.s**-2),log(cm.s**-2),log(cm.s**-2),dex,dex,dex,pc,pc,pc,mag,mag,mag,mag,mag,mag,mag,mag,mag,,
int64,object,int64,int64,float64,float64,float32,float64,float32,float64,float32,float32,float32,float64,float32,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int16,int16,int16,int16,float32,float32,float32,float32,int16,bool,float32,float32,float32,float32,float32,float32,float32,float32,int16,int16,float32,int16,int16,int16,float32,float32,int16,int16,float32,float32,float32,float32,float32,float32,float32,float32,float32,bool,int16,float64,float32,float32,float32,int16,float64,float32,float32,float32,int16,float64,float32,float32,float32,float32,int16,int16,int16,int16,int16,float32,float32,float32,float32,float32,int16,int16,int16,int16,float32,float32,float32,float32,float32,float32,float32,fl

Primary source_id=461701979234355968
Secondary source_id=461701979234355968


In [ ]:
# do a Gaia cone search for the primary and secondary
from astroquery.simbad import Simbad
import re

# extract the Gaia DR3 id from the Simbad ids string
def get_gaia_dr3_id(ids):
    regex = 'GAIA DR3 (\\d+)'
    result = re.search(regex, ids, re.IGNORECASE)

    return result.group(1) 


search_radius = 3*u.arcsec

Simbad.add_votable_fields('ids', 'velocity')#, 'typed_id')
results_pri = Simbad.query_region(c_pri_j2000, radius=search_radius)
#results_pri.sort('dist')
results_sec = Simbad.query_region(c_sec_j2000, radius=search_radius)

gaia_id_pri=get_gaia_dr3_id(results_pri['ids'][0])
gaia_id_sec=get_gaia_dr3_id(results_sec['ids'][0])
print(f"gaia_id_pri={gaia_id_pri}")
print(f"gaia_id_sec={gaia_id_sec}")

display(results_pri)
display(results_sec)
#results_t = Table()
#results_t = vstack([results_t, results])

print(c_pri_j2000.to_string())




AttributeError: 'NoneType' object has no attribute 'group'